# Lab 02A: Prompt Chaining + Code Migration (no API key)

**Week 1 — Agentic AI: Building Autonomous Intelligent Systems**

This is the keyless companion to the prompt-chaining lab. It uses Colab's **built-in AI** (`from google.colab import ai`) — so there is **no API key to create, no secret to configure, and nothing to install**. The trade-off is that this minimal SDK has no *guaranteed* structured-output mode, so we build robustness ourselves: every step that returns data goes through a **parse → validate → retry** helper.

You will build two chains:

**Part 1 — A research assistant** that chains LLM calls into a pipeline:
1. `summarize(text)` → 2. `extract_questions(summary)` → 3. `classify_questions(questions)` → 4. `research_pipeline(text)`.

**Part 2 — A code migrator** that moves **JavaScript → Python** through a language-neutral **IR** (intermediate representation):
`lift_to_ir` → `lower_to_python` → `review_migration`.

The big idea in both: instead of one giant, hard-to-debug prompt, **decompose the work into small, validatable links**.

```
  ONE GIANT PROMPT                          A CHAIN OF SMALL LINKS
  ────────────────                          ──────────────────────
  ┌──────────────────────────┐             ┌───────────┐  ┌───────────┐  ┌───────────┐
  │ "summarize AND extract   │   text ────▶│ summarize │─▶│ extract Q │─▶│ classify  │─▶ result
  │  questions AND classify  │             └───────────┘  └───────────┘  └───────────┘
  │  them all at once"       │                   │              │              │
  └──────────────────────────┘                   └── inspect & validate each handoff ──┘
           │
           ▼                                  when a link misbehaves, you see
  one blob out — when it's wrong,            exactly which one to fix (and only
  which part failed? you can't tell.         re-run that link, not the whole thing)
```

Part 2 adds a second lesson — *the IR is a reusable contract*. Each link talks to the IR rather than the raw source, so you can retarget the migration to another language by swapping only the final link.

> **Which SDK?** This lab uses `google.colab.ai`, the simplest on-ramp but deliberately limited. The API-key track (Labs 01A-sibling, 02+) uses the full **`google-genai`** SDK, which adds dedicated system instructions, *guaranteed* schema output, tool use, and async. We flag the differences as they come up.

## Setup: nothing to set up

Colab ships with an `ai` module that is already authenticated for you. Import it and call it. First, list the models you can call — names look like `google/...`.

In [93]:
from google.colab import ai

# See what you can call.
for name in ai.list_models():
    print(name)

google/gemini-2.5-flash-lite
google/gemini-3.5-flash


Pick a model and store it in a `MODEL` constant so the rest of the notebook is a one-line change, then run a quick connectivity check.

> If the next cell raises a *model not found* error, copy one of the names printed above into `MODEL`.

In [94]:
MODEL = "google/gemini-2.5-flash-lite"

# Quick connectivity check
print(ai.generate_text("Say 'Setup complete!' and nothing else.", model_name=MODEL))

Setup complete!


If you saw `Setup complete!`, you're connected to Gemini — no key, no install.

## Robust helpers: free text and *best-effort* structured JSON

`generate()` is for steps that just need prose back (a summary, an answer). We add a tiny retry so a transient hiccup pauses instead of crashing the whole chain.

`generate_json()` is the piece that makes chaining reliable on this minimal SDK. The full `google-genai` SDK lets you hand the model a **schema** and *guarantees* valid, typed JSON. `colab.ai` has no such mode — all we can do is **ask** for JSON and hope. So we harden it with a **parse → validate → retry** loop:

```
   prompt + "return ONLY JSON"
        │
        ▼
   ┌──────────────┐
   │  generate()  │◀───────────────────────────────┐
   └──────────────┘                                 │
        │ raw text                                   │  re-prompt, showing the model
        ▼                                            │  its bad output + the exact error
   strip ``` fences  →  json.loads  →  validate(schema)
        │                                  │         │
        │ ok                               │ JSONDecodeError /
        ▼                                  │ ValidationError ──┘
   typed Python object                     │
   (or plain parsed JSON)                  └── after `retries` failures ──▶ raise ValueError
```

The retry loop is the whole point: a single malformed reply doesn't break the chain — the model gets one more shot *with the error in hand*. This *approximates* the schema guarantee you'd get for free in the API-key track — which is exactly the motivation for upgrading later.

1. append a "return ONLY valid JSON" instruction,
2. strip any ```` ```json ```` / ```` ``` ```` code fences,
3. `json.loads` it — and, if a `pydantic` `schema` is given, **validate** against it,
4. on failure, **re-prompt with a repair message** that shows the model its bad output and the exact error, up to `retries` times,
5. raise a clear error if it still won't comply.

In [95]:
import json
import time
from typing import Literal

from pydantic import BaseModel, TypeAdapter, ValidationError


def generate(prompt: str, retries: int = 2) -> str:
    """Send a single prompt to Gemini via Colab's built-in AI and return the text.

    Retries with a short backoff on transient errors so a hiccup pauses the
    chain instead of crashing it.
    """
    for attempt in range(retries + 1):
        try:
            return ai.generate_text(prompt, model_name=MODEL).strip()
        except Exception as e:
            if attempt == retries:
                raise
            print(f"  Transient error ({e!r}); retrying in 5s...")
            time.sleep(5)


def _strip_fences(raw: str) -> str:
    """Remove ```json / ``` code fences the model often wraps JSON in."""
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[-1] if "\n" in raw else raw
        raw = raw.removeprefix("```json").removeprefix("```")
        raw = raw.removesuffix("```")
    return raw.strip()


def generate_json(prompt: str, schema=None, retries: int = 2):
    """Ask for JSON, strip fences, parse, optionally validate against a schema.

    `schema` may be a pydantic model or a typed container such as list[str] or
    list[MyModel]. On a parse/validation failure we re-prompt the model with its
    own bad output and the error, up to `retries` times. Returns parsed Python
    objects (typed instances when a pydantic schema is given).
    """
    adapter = TypeAdapter(schema) if schema is not None else None
    instruction = "\n\nReturn ONLY valid JSON — no markdown fences, no prose."
    current = prompt + instruction

    last_error = None
    for attempt in range(retries + 1):
        raw = generate(current)
        text = _strip_fences(raw)
        try:
            data = json.loads(text)
            return adapter.validate_python(data) if adapter else data
        except (json.JSONDecodeError, ValidationError) as e:
            last_error = e
            print(f"  Invalid JSON (attempt {attempt + 1}); asking the model to fix it...")
            current = (
                f"{prompt}{instruction}\n\n"
                f"Your previous output was invalid and could not be used:\n{raw}\n\n"
                f"The error was:\n{e}\n\n"
                "Return corrected JSON only."
            )
    raise ValueError(f"Could not get valid JSON after {retries + 1} attempts: {last_error}")

---

## Part 1 — A research-assistant chain

Each link's output becomes the next link's input. Steps that need *data* use `generate_json()` with a `pydantic` schema, so a malformed reply is caught and repaired rather than silently breaking the next step.

```
   text
    │
    ▼  summarize()            ── prose link, generate()
   "3-sentence summary"
    │
    ▼  extract_questions()    ── data link, generate_json(schema=list[str])
   ["Q1", "Q2", ... "Q5"]
    │
    ▼  classify_questions()   ── data link, generate_json(schema=list[ClassifiedQuestion])
   [{question, type}, ...]    ── type is locked to factual | analytical | creative
```

Notice the **handoff contract** tightens as you go: free prose → a list of strings → a list of typed objects. Each `generate_json` link validates the shape it receives the moment it's produced.

### Step 1 — Summarize

The first link. Its output (a 3-sentence summary) feeds the next step. Plain prose, so `generate()`.

> **Prompt design.** Every prompt in this lab follows the same Markdown structure — **Role · Context · Task · Constraints · Format**:
>
> | Section | What it does |
> | :--- | :--- |
> | **Role / Persona** | who the model should act as — always a *senior/expert* (sets the bar for quality) |
> | **Context** | the goal, the audience, and where this step sits in the pipeline |
> | **Task** | the one thing to do, stated plainly |
> | **Constraints** | the boundaries — limits, rules, what to avoid |
> | **Format** | exactly how to deliver the output |
>
> This is the single highest-leverage habit in prompt design. Skim the prompt below and you'll see the five sections.

In [96]:
def summarize(text: str) -> str:
    """Summarize the given text in exactly 3 sentences."""
    return generate(f"""# Role
You are a senior research analyst who distills dense material into precise executive briefings.

# Context
This summary is the first link in an automated research pipeline; later steps generate and classify follow-up questions from it, so it must faithfully capture the source's central argument.

# Task
Summarize the text in the Input section.

# Constraints
- Exactly 3 sentences — no more, no fewer.
- Capture the most important points; drop examples, caveats, and asides.
- Stay faithful to the source; add no opinions or outside facts.

# Format
Plain prose: three sentences in a single paragraph. No headings, lists, or preamble.

# Input
{text}""")

### Step 2 — Extract research questions

We want a **list of strings**, so we pass `schema=list[str]`. `generate_json` validates the shape and retries if the model returns prose; we slice to 5 in case it is generous.

In [97]:
def extract_questions(summary: str) -> list[str]:
    """Generate 5 follow-up research questions from the summary."""
    questions = generate_json(
        f"""# Role
You are a senior research strategist who designs rigorous lines of inquiry.

# Context
You receive a 3-sentence summary produced earlier in a research pipeline. Your questions are the next link: they will be classified and used to guide deeper investigation of the topic.

# Task
Generate exactly 5 follow-up research questions that would meaningfully deepen understanding of the topic.

# Constraints
- Exactly 5 questions.
- Each must be open-ended and answerable through research (no yes/no questions).
- Cover distinct angles; no duplicates or trivial rephrasings.
- Ground every question in the summary; do not introduce facts it does not imply.

# Format
A JSON array of exactly 5 strings, each one question. No numbering, no surrounding object.

# Input (summary)
{summary}""",
        schema=list[str],
    )
    return questions[:5]

### Step 3 — Classify questions

This is the step most likely to break with raw string parsing. The fix: describe the output as a **schema**. `ClassifiedQuestion` pins `type` to exactly three values via `Literal`, and `list[ClassifiedQuestion]` says "one per question." A prose reply or invalid label fails validation and triggers the repair retry.

In [98]:
class ClassifiedQuestion(BaseModel):
    question: str
    type: Literal["factual", "analytical", "creative"]


def classify_questions(questions: list[str]) -> list[ClassifiedQuestion]:
    """Classify each question as 'factual', 'analytical', or 'creative'."""
    questions_text = "\n".join(f"{i + 1}. {q}" for i, q in enumerate(questions))
    return generate_json(
        f"""# Role
You are a senior research methodologist who categorizes inquiry by the kind of thinking it demands.

# Context
These questions come from a research pipeline. Downstream steps treat factual, analytical, and creative questions differently, so each label must be accurate.

# Task
Classify every question in the Input section as exactly one of: factual, analytical, or creative.

# Constraints
- factual: has a specific, verifiable answer.
- analytical: requires reasoning, comparison, or synthesis.
- creative: open-ended, requires imagination or speculation.
- Classify each question exactly once and keep its text verbatim.

# Format
A JSON array of objects, one per question, each with `question` (string) and `type` (factual | analytical | creative).

# Input (questions)
{questions_text}""",
        schema=list[ClassifiedQuestion],
    )

### Step 4 — The full pipeline

Chain the three steps. Each `print` is a **checkpoint** — when something goes wrong you can see exactly which link produced bad output. That visibility is the whole point of chaining versus one giant prompt.

In [99]:
def research_pipeline(text: str) -> dict:
    """Chain all three steps: summarize -> extract questions -> classify."""
    print("Step 1: Summarizing...")
    summary = summarize(text)
    print(f"  Summary: {summary}")

    print("Step 2: Extracting research questions...")
    questions = extract_questions(summary)
    print(f"  Found {len(questions)} questions")

    print("Step 3: Classifying questions...")
    classified = classify_questions(questions)

    return {"summary": summary, "questions": classified}

### Run it — the research pipeline end to end

In [100]:
SAMPLE_TEXT = """
Large language models (LLMs) have demonstrated remarkable capabilities across a wide range
of tasks, from creative writing to complex reasoning. However, a key limitation is their
inability to take actions in the world or access up-to-date information. Agentic AI systems
address this by equipping LLMs with tools — functions the model can call to search the web,
execute code, read files, or interact with APIs. Combined with planning mechanisms like
ReAct (Reason + Act), these systems can autonomously decompose complex goals into steps,
execute each step using appropriate tools, and adapt based on the results. This shift from
passive text generation to active goal pursuit represents one of the most significant
developments in applied AI, enabling applications in software engineering, scientific
research, business automation, and beyond.
"""

result = research_pipeline(SAMPLE_TEXT)

print("\nFinal Result:")
print(f"Summary:\n  {result['summary']}\n")
print("Classified Questions:")
for item in result["questions"]:
    print(f"  [{item.type.upper():10s}] {item.question}")

Step 1: Summarizing...
  Summary: Agentic AI systems empower large language models with tools and planning mechanisms, such as ReAct, to overcome their limitations in acting on the world and accessing current information. By enabling LLMs to autonomously decompose complex goals into executable steps using various tools, these systems transition from passive text generation to active goal pursuit. This advancement is driving significant progress in applied AI across diverse fields like software engineering and scientific research.
Step 2: Extracting research questions...
  Found 5 questions
Step 3: Classifying questions...

Final Result:
Summary:
  Agentic AI systems empower large language models with tools and planning mechanisms, such as ReAct, to overcome their limitations in acting on the world and accessing current information. By enabling LLMs to autonomously decompose complex goals into executable steps using various tools, these systems transition from passive text generation to

---

## Part 2 — Code migration via an **IR**

Now a richer chain that does something genuinely useful: migrate **JavaScript → Python**. A naive approach is one prompt — *"translate this code to Python"* — but that is a black box: when it gets something subtly wrong, you have no idea which part of its reasoning failed, and you cannot reuse any of the work for a different target language.

Instead we route the migration through a language-neutral **intermediate representation (IR)**:

| Stage | What it is |
| :--- | :--- |
| **IR** (Intermediate Representation) | a *language-neutral, semantic* description of what the code does — intent, params, returns, ordered steps |
| **Target code** | idiomatic code in the destination language, generated **from the IR, not the raw source** |

The chain reads top to bottom:

```
   JavaScript source
        │
        │  lift_to_ir         ─── LLM + schema   (describe what the code DOES, language-neutrally)
        ▼
   IR  (language-neutral: intent · params · returns · ordered steps)
        │
        │  lower_to_python    ─── LLM   (reads the IR, NOT the original JavaScript)
        ▼
   Python
        │
        │  review_migration   ─── LLM + schema   (self-check: source vs. target)
        ▼
   parity findings  [{severity, description, suggestion}, ...]
```

The point: each link is small and inspectable, and the **IR is the migration contract** — because `lower_to_python` reads the IR rather than the JavaScript, you can retarget to Go, Rust, or TypeScript by swapping only that one link.

### Link 1 — `lift_to_ir_python` (LLM, structured + validated)

The first link **lifts** the JavaScript source into a language-neutral **IR**: what each function *means* — its intent, parameters, return value, and the ordered steps it performs — described independently of JavaScript syntax. We pin the shape with a `pydantic` schema so `generate_json` validates and repairs it.

In [101]:
class IRStep(BaseModel):
    """One semantic operation inside a function, described language-neutrally."""
    description: str
    kind: Literal["init", "loop", "conditional", "call", "assignment", "return", "other"]


class IRFunction(BaseModel):
    name: str
    intent: str
    params: list[str]
    returns: str
    steps: list[IRStep]
    side_effects: list[str]


class IRProgram(BaseModel):
    summary: str
    functions: list[IRFunction]


SAMPLE_CODE = """
function fizzbuzz(n) {
  const results = [];
  for (let i = 1; i <= n; i++) {
    if (i % 15 === 0) {
      results.push("FizzBuzz");
    } else if (i % 3 === 0) {
      results.push("Fizz");
    } else if (i % 5 === 0) {
      results.push("Buzz");
    } else {
      results.push(String(i));
    }
  }
  return results;
}
"""
SAMPLE_PY_CODE = """
def fizzbuzz(n: int) -> list[str]:
    results = []
    for i in range(1, n + 1):
        if i % 15 == 0:
            results.append("FizzBuzz")
        elif i % 3 == 0:
            results.append("Fizz")
        elif i % 5 == 0:
            results.append("Buzz")
        else:
            results.append(str(i))
    return results
    """

SAMPLE_MATH_JAVASCRIPT_CODE="""
function intDivide(a, b) {
    return Math.floor(13 / 5);
}
"""

def lift_to_ir_python(code: str) -> IRProgram:
    """Lift JavaScript source into a language-neutral intermediate representation."""
    return generate_json(
        f"""# Role
You are a senior compiler engineer who specializes in language-agnostic intermediate representations.

# Context
You are the front-end of a source-to-source migration pipeline. A later back-end lowers your IR into another language, so the IR must describe what the code DOES, not how JavaScript spells it.

# Task
Lift the JavaScript program in the Input section into a language-neutral intermediate representation (IR).

# Constraints
- Describe semantics — intent, parameters, return value, ordered steps — not JavaScript-specific syntax.
- Capture every function in the source; do not invent functions that are not there.
- Keep each step implementation-neutral so any target language could realize it.


# Format
Return ONE JSON object with exactly two keys, `summary` and `functions`, and nothing else. Follow these field rules precisely:
- `summary`: a single plain string (one sentence describing the whole program). NOT an object.
- each item in `functions` has keys `name`, `intent`, `params`, `returns`, `steps`.
- `params`: a JSON array of plain strings — parameter NAMES only, e.g. ["n"]. Never objects like {{"name": "n", "type": "integer"}}.
- `returns`: a plain string, e.g. "a list of strings". Never an object like {{"type": "array"}}.
- `steps`: an array of objects, each exactly {{"description": "<text>", "kind": "<label>"}}.
- `side_effects`: a JSON array of plain strings, e.g. "prints to stdout", "mutates input".
- `kind` MUST be exactly one of: init, loop, conditional, call, assignment, return, other. Do NOT invent labels — map any if/else branching to "conditional".

# Input (JavaScript source)
{code}""",
        schema=IRProgram,
    )

print("================Javascript to Python===================")
ir = lift_to_ir_python(SAMPLE_CODE)
print("Program summary:", ir.summary, "\n")
for fn in ir.functions:
    print(f"- {fn.name}({', '.join(fn.params)}) -> {fn.returns}")
    print(f"    intent: {fn.intent}")
    for step in fn.steps:
        print(f"    [{step.kind}] {step.description}")

def lift_to_ir_math(code: str) -> IRProgram:
    """Lift JavaScript source into a language-neutral intermediate representation."""
    return generate_json(
        f"""# Role
You are a senior compiler engineer who specializes in language-agnostic intermediate representations.

# Context
You are the front-end of a source-to-source migration pipeline. A later back-end lowers your IR into another language, so the IR must describe what the code DOES, not how JavaScript spells it.

# Task
Lift the JavaScript program in the Input section into a language-neutral intermediate representation (IR).

# Constraints
- Describe semantics — intent, parameters, return value, ordered steps — not JavaScript-specific syntax.
- Capture every function in the source; do not invent functions that are not there.
- Keep each step implementation-neutral so any target language could realize it.


# Format
Return ONE JSON object with exactly two keys, `summary` and `functions`, and nothing else. Follow these field rules precisely:
- `summary`: a single plain string (one sentence describing the whole program). NOT an object.
- each item in `functions` has keys `name`, `intent`, `params`, `returns`, `steps`.
- `params`: a JSON array of plain strings — parameter NAMES only, e.g. ["n"]. Never objects like {{"name": "n", "type": "integer"}}.
- `returns`: a plain string, e.g. "a list of strings". Never an object like {{"type": "array"}}.
- `steps`: an array of objects, each exactly {{"description": "<text>", "kind": "<label>"}}.
- `side_effects`: a JSON array of plain strings, e.g. "prints to stdout", "mutates input".
- `kind` MUST be exactly one of: init, loop, conditional, call, assignment, return, other. Do NOT invent labels — map any if/else branching to "conditional".

# Input (JavaScript source)
{code}""",
        schema=IRProgram,
    )

print("==============Javascript to Python Math Function==============")
ir_math = lift_to_ir_math(SAMPLE_MATH_JAVASCRIPT_CODE)
print("Program summary:", ir_math.summary, "\n")
for fn in ir_math.functions:
    print(f"- {fn.name}({', '.join(fn.params)}) -> {fn.returns}")
    print(f"    intent: {fn.intent}")
    for step in fn.steps:
        print(f"    [{step.kind}] {step.description}")




================Javascript to Python===================
Program summary: Generates a list of strings based on the FizzBuzz sequence up to a given number n. 

- fizzbuzz(n) -> a list of strings
    intent: Generates a FizzBuzz sequence up to a given number.
    [assignment] Initialize an empty list to store results.
    [loop] Loop from 1 up to and including n.
    [conditional] Check if the current number is divisible by 15.
    [call] Add 'FizzBuzz' to the results list.
    [conditional] Check if the current number is divisible by 3.
    [call] Add 'Fizz' to the results list.
    [conditional] Check if the current number is divisible by 5.
    [call] Add 'Buzz' to the results list.
    [call] Convert the current number to a string and add it to the results list.
    [return] Return the list of results.
==============Javascript to Python Math Function==============
  Invalid JSON (attempt 1); asking the model to fix it...
Program summary: Defines a function 'intDivide' that returns a h

###Link 1.1- lift_to_js_ir(LLM,structured + validated) (Python to JS)
Javascript new IR creation.



In [102]:
def lift_to_js_ir(code: str) -> IRProgram:
    """Lift Python source into a language-neutral intermediate representation."""
    return generate_json(
        f"""# Role
You are a senior compiler engineer who specializes in language-agnostic intermediate representations.

# Context
You are the front-end of a source-to-source migration pipeline. A later back-end lowers your IR into another language, so the IR must describe what the code DOES, not how Python spells it.

# Task
Lift the Python program in the Input section into a language-neutral intermediate representation (IR).

# Constraints
- Describe semantics — intent, parameters, return value, ordered steps — not Python-specific syntax.
- Capture every function in the source; do not invent functions that are not there.
- Keep each step implementation-neutral so any target language could realize it.

# Format
Return ONE JSON object with exactly two keys, `summary` and `functions`, and nothing else. Follow these field rules precisely:
- `summary`: a single plain string (one sentence describing the whole program). NOT an object.
- each item in `functions` has keys `name`, `intent`, `params`, `returns`, `steps`.
- `params`: a JSON array of plain strings — parameter NAMES only, e.g. ["n"]. Never objects like {{"name": "n", "type": "integer"}}.
- `returns`: a plain string, e.g. "a list of strings". Never an object like {{"type": "array"}}.
- `steps`: an array of objects, each exactly {{"description": "<text>", "kind": "<label>"}}.
- `side_effects`: a JSON array of plain strings, e.g. "prints to stdout", "mutates input".
- `kind` MUST be exactly one of: init, loop, conditional, call, assignment, return, other. Do NOT invent labels — map any if/else branching to "conditional".

# Input (Python source)
{code}""",
        schema=IRProgram,
    )

print("================Python to Javascript===================")
ir_js = lift_to_js_ir(SAMPLE_PY_CODE)
print("Program summary:", ir_js.summary, "\n")
for fn in ir_js.functions:
    print(f"- {fn.name}({', '.join(fn.params)}) -> {fn.returns}")
    print(f"    intent: {fn.intent}")
    for step in fn.steps:
        print(f"    [{step.kind}] {step.description}")

================Python to Javascript===================
  Invalid JSON (attempt 1); asking the model to fix it...
Program summary: Generates a list of strings representing numbers from 1 to n, with multiples of 3 replaced by 'Fizz', multiples of 5 by 'Buzz', and multiples of 15 by 'FizzBuzz'. 

- fizzbuzz(n) -> a list of strings
    intent: Generates a list of strings according to FizzBuzz rules up to a given number.
    [init] Initialize an empty list to store results.
    [loop] Iterate through numbers from 1 up to and including n.
    [conditional] Check if the current number is divisible by 15.
    [assignment] If divisible by 15, append 'FizzBuzz' to the results list.
    [conditional] If not divisible by 15, check if the current number is divisible by 3.
    [assignment] If divisible by 3, append 'Fizz' to the results list.
    [conditional] If not divisible by 3, check if the current number is divisible by 5.
    [assignment] If divisible by 5, append 'Buzz' to the results list.

### Link 2 — `lower_to_python` (LLM, prose/code) (JS to Python) FizzBuzz

We **lower** the IR into idiomatic Python. The model works from the **IR, not the raw JavaScript** — the IR is the migration *contract*. Swap the target instruction and the same IR could lower to Go, Rust, or TypeScript without re-reading the source.

In [103]:
def lower_to_python(ir: IRProgram) -> str:
    """Generate idiomatic Python from the language-neutral IR."""
    return generate(f"""# Role
You are a senior Python engineer who writes clean, idiomatic, modern Python (3.10+).

# Context
You are the back-end of a migration pipeline. You receive a language-neutral IR — not the original source — describing each function's intent and ordered steps. Faithfully implementing the IR is what preserves behavioral parity with the original program.

# Task
Generate Python that implements the intermediate representation in the Input section.

# Constraints
- Match each function's intent and ordered steps exactly.
- Use modern, idiomatic Python (list comprehensions, f-strings, and clear names where natural).
- Implement only what the IR specifies; introduce no extra behavior.
- CRITICAL ARITHMETIC CHECK: Look explicitly at division operations. JavaScript uses `Math.floor(a / b)` for integer division. If Python uses a single slash `/` instead of a double slash `//`, it will cause a critical floating-point type drift bug.

# Format
Output ONLY the Python, in a single ```python code block, with no commentary before or after.

# Input (IR)
{ir.model_dump_json(indent=2)}""")

py_code = lower_to_python(ir)
print(py_code)

```python
def fizzbuzz(n: int) -> list[str]:
    """
    Generates a FizzBuzz sequence up to a given number.
    """
    results = []
    for i in range(1, n + 1):
        if i % 15 == 0:
            results.append("FizzBuzz")
        elif i % 3 == 0:
            results.append("Fizz")
        elif i % 5 == 0:
            results.append("Buzz")
        else:
            results.append(str(i))
    return results
```


### Link 2.1 — `lower_to_math_python` (LLM, prose/code) (JS to Python)  Math.floor(a / b) for integer division

We **lower** the IR into idiomatic Python. The model works from the **IR, not the raw JavaScript** — the IR is the migration *contract*. By merging two separate JSON strings into one prompt block, the final string will no longer be a single valid JSON payload. This will deeply confuse the LLM and might cause hallucination.

In [104]:
def lower_to_math_python(ir_math: IRProgram) -> str:
    """Generate idiomatic Python from the language-neutral IR."""
    return generate(f"""# Role
You are a senior Python engineer who writes clean, idiomatic, modern Python (3.10+).

# Context
You are the back-end of a migration pipeline. You receive a language-neutral IR — not the original source — describing each function's intent and ordered steps. Faithfully implementing the IR is what preserves behavioral parity with the original program.

# Task
Generate Python that implements the intermediate representation in the Input section.

# Constraints
- Match each function's intent and ordered steps exactly.
- Use modern, idiomatic Python (list comprehensions, f-strings, and clear names where natural).
- Implement only what the IR specifies; introduce no extra behavior.
- CRITICAL ARITHMETIC CHECK: Look explicitly at division operations. JavaScript uses `Math.floor(a / b)` for integer division. If Python uses a single slash `/` instead of a double slash `//`, it will cause a critical floating-point type drift bug.

# Format
Output ONLY the Python, in a single ```python code block, with no commentary before or after.

# Input (IR)
{ir_math.model_dump_json(indent=2)}""")


py_math_code = lower_to_math_python(ir_math)
print(py_math_code)

```python
def intDivide(a, b):
    """
    Performs a division operation and returns the floored result of a specific calculation.

    Args:
        a: Unused parameter.
        b: Unused parameter.

    Returns:
        The floored result of 13 divided by 5.
    """
    # Calculate the floor of 13 divided by 5.
    result = 13 // 5
    # Return the computed value.
    return result
```


### Link 2.2 — `lower_to_go` (LLM, prose/code) (JS to GO)

We **lower** the IR into idiomatic Go. The model works from the **IR, not the raw JavaScript** — the IR is the migration *contract*. Swap the target instruction and the same IR could lower to Go, Rust, or TypeScript without re-reading the source.

In [105]:
def lower_to_go(ir: IRProgram) -> str:
    """Generate idiomatic Go from the language-neutral IR."""
    return generate(f"""# Role
You are a senior Go engineer who writes clean, idiomatic, modern Go (1.26.5).

# Context
You are the back-end of a migration pipeline. You receive a language-neutral IR — not the original source — describing each function's intent and ordered steps. Faithfully implementing the IR is what preserves behavioral parity with the original program.

# Task
Generate Go that implements the intermediate representation in the Input section.

# Constraints
- Match each function's intent and ordered steps exactly.
- Use modern, idiomatic Go (list comprehensions, f-strings, and clear names where natural).
- Implement only what the IR specifies; introduce no extra behavior.

# Format
Output ONLY the Go, in a single ```Go code block, with no commentary before or after.

# Input (IR)
{ir.model_dump_json(indent=2)}""")


go_code = lower_to_go(ir)
print(go_code)

```go
package main

import (
	"strconv"
)

func fizzbuzz(n int) []string {
	results := []string{}
	for i := 1; i <= n; i++ {
		if i%15 == 0 {
			results = append(results, "FizzBuzz")
		} else if i%3 == 0 {
			results = append(results, "Fizz")
		} else if i%5 == 0 {
			results = append(results, "Buzz")
		} else {
			results = append(results, strconv.Itoa(i))
		}
	}
	return results
}
```


### Link 2.3 — `lower_to_javascript` (LLM, prose/code)(Python to JS)

We **lower** the IR into idiomatic Javascript. The model works from the **IR, not Python** — the IR is the migration *contract*. Swap the target instruction and the same IR could lower to Go, Rust, or TypeScript without re-reading the source.

In [106]:
def lower_to_javascript(ir_js: IRProgram) -> str:
    """Generate idiomatic Javascript from the language-neutral IR."""
    return generate(f"""# Role
You are a senior Javascript engineer who writes clean, idiomatic, modern ECMAScript 2026 (ES17).

# Context
You are the back-end of a migration pipeline. You receive a language-neutral IR — not the original source — describing each function's intent and ordered steps. Faithfully implementing the IR is what preserves behavioral parity with the original program.

# Task
Generate Javascript that implements the intermediate representation in the Input section.

# Constraints
- Match each function's intent and ordered steps exactly.
- Use modern, idiomatic Javascript (list comprehensions, f-strings, and clear names where natural).
- Implement only what the IR specifies; introduce no extra behavior.

# Format
Output ONLY the Javascript, in a single ```Javascript code block, with no commentary before or after.

# Input (IR)
{ir_js.model_dump_json(indent=2)}""")


js_code = lower_to_javascript(ir_js)
print(js_code)

```javascript
function fizzbuzz(n) {
  const results = [];
  for (let i = 1; i <= n; i++) {
    if (i % 15 === 0) {
      results.push('FizzBuzz');
    } else if (i % 3 === 0) {
      results.push('Fizz');
    } else if (i % 5 === 0) {
      results.push('Buzz');
    } else {
      results.push(i.toString());
    }
  }
  return results;
}
```


### Link 3 — `review_migration` (LLM, structured self-check) (JS to Python)

The last link makes the chain **check its own work**. It compares the original JavaScript with the generated Python and reports parity issues as structured findings, each with a severity. Because the output is schema-validated, a downstream gate can act on it programmatically (see the exercises).

In [107]:
class ParityIssue(BaseModel):
    severity: Literal["low", "medium", "high"]
    description: str
    suggestion: str


def review_migration(js_src: str, python_src: str) -> list[ParityIssue]:
    """Compare source and target, reporting behavioral-parity issues."""
    return generate_json(
        f"""# Role
You are a senior software engineer performing a careful cross-language code review.

# Context
A program was migrated from JavaScript to Python through an IR. Subtle cross-language differences — number types and division, truthiness, equality (=== vs ==), array vs list semantics, string coercion — are the usual source of migration bugs and must be caught before this ships.

# Task
Compare the original JavaScript with the migrated Python in the Input section and report every place their runtime behavior could differ.

# Constraints
- Report only genuine behavioral-parity risks, not style preferences.
- Weigh the classic traps: numeric types and division, truthiness, equality and coercion, mutation, off-by-one and iteration bounds.
- If the two are behaviorally equivalent, return an empty array.

# Format
A JSON array of objects, each with `severity` (low | medium | high), `description`, and `suggestion`.

# Input
## JavaScript
{js_src}

## Python
{python_src}""",
        schema=list[ParityIssue],
    )


issues = review_migration(SAMPLE_CODE, py_code)
if not issues:
    print("No parity issues found.")
for issue in issues:
    print(f"[{issue.severity.upper():6s}] {issue.description}")
    print(f"         fix: {issue.suggestion}")

issues_math = review_migration(SAMPLE_MATH_JAVASCRIPT_CODE, py_code)
if not issues:
    print("No parity issues found.")
for issue in issues_math:
    print(f"[{issue.severity.upper():6s}] {issue.description}")
    print(f"         fix: {issue.suggestion}")



[LOW   ] JavaScript's `let` keyword allows for block-scoped variables, while Python's `for` loop variable `i` is local to the loop. This is generally equivalent for this specific loop structure, but in more complex scenarios, JavaScript's scoping rules can differ significantly from Python's. However, for this direct translation, the behavior is effectively the same.
         fix: No change needed. The behavior is equivalent in this context.
[LOW   ] JavaScript's `String(i)` explicitly converts the number `i` to a string. Python's `str(i)` performs a similar explicit conversion. The behavior for integer to string conversion is consistent between the two languages.
         fix: No change needed. The behavior is equivalent.
[LOW   ] JavaScript's `Array.prototype.push()` and Python's `list.append()` both add elements to the end of their respective collections. The behavior is functionally identical for this use case.
         fix: No change needed. The behavior is equivalent.
[LOW   ] Jav

### Link 3.1 — `reverse_review_migration` (LLM, structured self-check) (Python to JS)

The last link makes the chain **check its own work**. It compares the original Python with the generated Javascript and reports parity issues as structured findings, each with a severity. Because the output is schema-validated, a downstream gate can act on it programmatically (see the exercises).

In [108]:
def reverse_review_migration(python_src: str, js_src: str) -> list[ParityIssue]:
    """Compare source and target, reporting behavioral-parity issues."""
    return generate_json(
        f"""# Role
You are a senior software engineer performing a careful cross-language code review.

# Context
A program was migrated from Python to Javascript through an IR. Subtle cross-language differences — number types and division, truthiness, equality (=== vs ==), array vs list semantics, string coercion — are the usual source of migration bugs and must be caught before this ships.

# Task
Compare the original Python with the migrated Javascript in the Input section and report every place their runtime behavior could differ.

# Constraints
- Report only genuine behavioral-parity risks, not style preferences.
- Weigh the classic traps: numeric types and division, truthiness, equality and coercion, mutation, off-by-one and iteration bounds.
- If the two are behaviorally equivalent, return an empty array.

# Format
A JSON array of objects, each with `severity` (low | medium | high), `description`, and `suggestion`.

# Input
## Python
{python_src}

## JavaScript

{js_src}""",
        schema=list[ParityIssue],
    )


issues = reverse_review_migration(SAMPLE_PY_CODE, js_code)
if not issues:
    print("No parity issues found.")
for issue in issues:
    print(f"[{issue.severity.upper():6s}] {issue.description}")
    print(f"         fix: {issue.suggestion}")

[LOW   ] Python's `range(1, n + 1)` generates integers. JavaScript's `for` loop with `let i = 1; i <= n; i++` also generates integers. The iteration bounds and types of `i` appear to be equivalent in this specific case.
         fix: No action needed for this specific code, but be mindful of potential differences in other scenarios.
[LOW   ] Both Python and JavaScript use the modulo operator (`%`) for checking divisibility. For positive integers, their behavior is equivalent. The `===` strict equality check in JavaScript is appropriate here as the modulo operation will yield integer results.
         fix: No action needed.
[LOW   ] Python's `list.append()` and JavaScript's `Array.prototype.push()` both add elements to the end of their respective collections. The `results` array in JavaScript is initialized as an empty array, mirroring Python's `results = []`.
         fix: No action needed.
[LOW   ] Python's `str(i)` converts an integer to its string representation. JavaScript's `i.toS

###Link 4 - drift Report

In [109]:
def drift_report(SAMPLE_CODE: str, js_src: str) -> dict:
    """Compare the initial JS and final JS to analyze what drifted and why, returning a dictionary."""
    return generate_json(
        f"""# Role
You are a senior data analyst/engineer tool.

# Task
Compare the original JavaScript code with the round-tripped JavaScript code. Identify code drift and logical drift.

# Constraints
- Check if loops changed types (e.g., standard loop vs array methods).
- Check if incorrect typing can cause syntax errors.
- Find out which step in the chain may likely cause the drift.

# Format
Return ONE JSON object with exactly the following keys:
- "has_drift": true/false (boolean)
- "code_difference": "description of code changes" (string)
- "logic_changes": "description of logic or data type changes" (string)
- "responsible_link": exactly one of "lift_to_ir", "lift_to_js_ir","lower_to_python", "lower_to_javascript", "review_migration","reverse_review_migration" or "none" (string)
# Input
## Original JavaScript
{SAMPLE_CODE}

## Round-Tripped JavaScript
{js_src}"""
    )

drift = drift_report(SAMPLE_CODE,js_code)

print("\n=== Drift Report ===")
print(f"Has Drift: {drift['has_drift']}")
print(f"Responsible Link: {drift['responsible_link']}")
print(f"Code Difference: {drift['code_difference']}")
print(f"Logic Changes: {drift['logic_changes']}")


=== Drift Report ===
Has Drift: True
Responsible Link: none
Code Difference: The only code difference is the method used to convert the number to a string: `String(i)` in the original code versus `i.toString()` in the round-tripped code.
Logic Changes: There is no logic drift. Both `String(i)` and `i.toString()` achieve the same result of converting a number to its string representation in JavaScript, so the behavior of the function remains identical. There are no changes in loop types or potential typing issues that would cause syntax errors.


### The full migration pipeline

Chain the three links with checkpoints — the same pattern as Part 1. Each `print` shows which link is running, so a bad migration is easy to localize.

In [112]:
def migrate_pipeline(code: str) -> dict:
    """Chain: lift_to_ir -> lower_to_python -> review_migration."""
    print("Link 1: Lifting to IR (LLM)...")
    ir = lift_to_ir_python(code)
    print(f"  IR: {ir.summary}")

    ir_math_py = lift_to_ir_math(code)
    print(f"  IR: {ir_math_py.summary}")

    ir_js=lift_to_js_ir(code)
    print(f"  IR: {ir_js.summary}")

    print("Link 2: Lowering to Python (LLM)...")
    py_src = lower_to_python(ir)
    print(f"  Generated {len(py_src.splitlines())} lines of Python")

    print("Link 2: Lowering to Python (LLM) for Math.floor...")
    py_math_src = lower_to_math_python(ir_math)
    print(f"  Generated {len(py_math_src.splitlines())} lines of Python")

    print("Link 2.1: Lowering to Go (LLM)...")
    go_src = lower_to_go(ir)
    print(f"  Generated {len(go_src.splitlines())} lines of Go")

    print("Link 2.2: Lowering to Javascript (LLM)...")
    js_src = lower_to_javascript(ir_js)
    print(f"  Generated {len(js_src.splitlines())} lines of Javascript")

    print("Link 3: Reviewing javascript to python migration (LLM)...")
    issues_py = review_migration(code, py_src)
    print(f"  {len(issues)} parity issue(s) flagged")

    count = 2
    while any(i.severity == "high" for i in issues_py) and count > 0:
      print(f"--- Fixing high-severity issue (Count remaining: {count}) ---")
      py_src = lower_to_python(ir)
      issues_py = review_migration(code, py_src)
      count -= 1

    print("Link 3.1: Reviewing python to javascript migration (LLM)...")
    issues_js = reverse_review_migration(code, js_src)
    print(f"  {len(issues)} parity issue(s) flagged")


    return {"ir": ir, "python": py_src, "issues_py": issues_py ,"ir_js": ir_js, "javascript": js_src, "issues_js": issues_js}

python_migration = migrate_pipeline(SAMPLE_CODE)
print("\n=== Migrated Python ===")
print(python_migration["python"])

javascript_migration = migrate_pipeline(SAMPLE_PY_CODE)
print("\n=== Migrated Javascript ===")
print(javascript_migration["javascript"])






Link 1: Lifting to IR (LLM)...
  Invalid JSON (attempt 1); asking the model to fix it...
  IR: Generates a list of strings representing the FizzBuzz sequence up to a given number.
  Invalid JSON (attempt 1); asking the model to fix it...
  IR: The fizzbuzz function generates a list of strings representing numbers from 1 to n, applying specific string replacements for multiples of 3, 5, and 15.
  IR: Generates a list of strings representing numbers from 1 to n, with multiples of 3 replaced by 'Fizz', multiples of 5 by 'Buzz', and multiples of both by 'FizzBuzz'.
Link 2: Lowering to Python (LLM)...
  Generated 17 lines of Python
Link 2: Lowering to Python (LLM) for Math.floor...
  Generated 17 lines of Python
Link 2.1: Lowering to Go (LLM)...
  Generated 24 lines of Go
Link 2.2: Lowering to Javascript (LLM)...
  Generated 19 lines of Javascript
Link 3: Reviewing javascript to python migration (LLM)...
  4 parity issue(s) flagged
Link 3.1: Reviewing python to javascript migration (LLM)...

## Your turn (exercises)

1. **Validation gate.** After `migrate_pipeline`, if `review_migration` flags any `high`-severity issue, feed those findings back into `lower_to_python` (append them to the prompt) and regenerate — up to 2 times. This turns the self-check into a self-*repair* loop.
2. **Second backend, same IR.** Write `lower_to_go(ir)` (or `lower_to_typescript`) that reuses the *exact same* `IRProgram`. Notice you never re-read the source — the IR is the contract. This is why lowering through an IR beats one-shot translation.
3. **Richer IR.** Extend `IRFunction` with a `side_effects: list[str]` field (e.g. "prints to stdout", "mutates input"). You only change the schema — `generate_json` enforces it.
4. **Round-trip.** Feed the generated Python back through a Python-to-JavaScript version of the chain and diff the result against the original JavaScript. What drifts, and which link is responsible?
5. **Break it on purpose.** Feed `migrate_pipeline` JavaScript that uses `Math.floor(a / b)` for integer division. Does the Python come back using `//`? If `review_migration` misses the difference, strengthen its prompt.

When you're done, save a copy (**File → Save a copy in Drive**) and submit your notebook link via Canvas.

---

> **Robustness recap.** With `colab.ai` there is no schema *guarantee* — we approximated one with `generate_json`'s parse-validate-retry loop. The API-key track replaces the retry loop with `google-genai`'s native schema mode, where valid typed output is guaranteed on the first try. Same chaining concepts; sturdier plumbing.